In [ ]:
git # Install necessary libraries for fine-tuning. This ensures we have the latest versions.
# Note: On Kaggle, some packages might already be installed, but this ensures compatibility.
!pip install --no-cache-dir -U transformers datasets peft trl accelerate bitsandbytes evaluate scikit-learn matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 204.7 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 193.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 358.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.5/532.5 kB 367.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 360.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 315.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.4.2
    Uninstalling datasets-4.4.2:
      Successfully uninstalled datasets-4.4.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.

In [2]:
# 2. Dataset Loading & Inspection

import json
import pandas as pd
from pathlib import Path

# Path to dataset
# dataset_path = Path('../data/dataset.jsonl')
dataset_path = Path('/kaggle/input/major-manim-dataset-1-20/dataset.jsonl')

# Read JSONL dataset (skip non-JSON lines)
samples = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or not line.startswith('{'):
            continue  # skip comments, blank lines, or metadata
        try:
            samples.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("Skipping line due to JSON error:", e)

# Convert to DataFrame for inspection
sample_df = pd.DataFrame(samples)
print('Columns:', sample_df.columns.tolist())
print('Sample rows:')
display(sample_df.head())

# Identify input/output columns
input_col = None
output_col = None
for col in sample_df.columns:
    if 'prompt' in col or 'desc' in col:
        input_col = col
    if 'code' in col:
        output_col = col
print(f"Input column: {input_col}")
print(f"Output column: {output_col}")

Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due t

,prompt,code
0,show the integral of y = 1/x shaded from x=1 t...,from manim import *\nimport numpy as np\n\ncla...
1,"show signed area under y = x on [-1, 1].",from manim import *\n\nclass SignedAreaLinear(...
2,"compare exact area under y = x^2 on [0,1] with...",from manim import *\n\nclass CompareExactEstim...
3,show midpoint Riemann sum approximation for y ...,from manim import *\nimport numpy as np\n\ncla...
4,Visualize and compute the area between y = x a...,from manim import *\nimport numpy as np\n\ncla...


Input column: prompt
Output column: code


In [ ]:
# 3. Preprocessing: Instruction-Response Formatting

from datasets import Dataset

# Use the columns detected above
def format_instruction(example):
    # SFTTrainer expects a dict with 'prompt' and 'completion' keys by default
    # Truncate to prevent long sequences that cause OOM
    prompt = example[input_col].strip()[:512]  # Limit prompt length
    completion = example[output_col].strip()[:1024]  # Limit completion length
    return {
        'prompt': prompt,
        'completion': completion,
        'text': f"<s>[INST] {prompt} [/INST]\n{completion}</s>"
    }

# Apply formatting
dataset = Dataset.from_pandas(sample_df)
dataset = dataset.map(format_instruction)

# Show a sample
print('Formatted sample:')
print(dataset[0])

Map:   0%|          | 0/367 [00:00<?, ? examples/s]

Formatted sample:
{'prompt': 'show the integral of y = 1/x shaded from x=1 to x=e.', 'code': 'from manim import *\nimport numpy as np\n\nclass IntegralOneOverX(Scene):\n    def construct(self):\n        axes = Axes(\n            x_range=[0.5, np.e + 1, 0.5],\n            y_range=[0, 1.5, 0.5],\n            x_length=8,\n            y_length=5,\n            axis_config={"include_numbers": True}\n        ).to_edge(DOWN)\n\n        curve = axes.plot(lambda x: 1 / x, color=BLUE)\n        label = MathTex(r"y = \\frac{1}{x}").next_to(curve, UP + RIGHT)\n\n        # Shade area under curve from x=1 to x=e\n        area = axes.get_area(curve, x_range=[1, np.e], color=GREEN, opacity=0.5)\n\n        self.play(Create(axes), Create(curve), Write(label))\n        self.wait(1)\n        self.play(FadeIn(area))\n        self.wait(3)', 'completion': 'from manim import *\nimport numpy as np\n\nclass IntegralOneOverX(Scene):\n    def construct(self):\n        axes = Axes(\n            x_range=[0.5, np.e + 

In [ ]:
# 4. Model & Tokenizer Loading (Quantized)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Load model (quantized)
print('Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing for memory savings
model.gradient_checkpointing_enable()

# Enable flash attention if available
try:
    model.enable_flash_attn()
    print('Flash attention enabled!')
except Exception:
    print('Flash attention not available.')

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

In [ ]:
# 5. LoRA (QLoRA) & Trainer Setup
# LoRA (Low-Rank Adaptation) makes fine-tuning efficient by training only small adapters instead of the full model.
# This reduces memory usage and speeds up training.

from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# LoRA config (QLoRA, memory efficient)
lora_config = LoraConfig(
    r=16,  # Rank of the low-rank matrices
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.05,  # Dropout for regularization
    bias="none",  # No bias training
    task_type="CAUSAL_LM",  # For language models
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers to adapt
)

# Output dir for LoRA adapters (adjusted for Kaggle)
output_dir = "/kaggle/working/outputs/mistral/lora"
os.makedirs(output_dir, exist_ok=True)

# Training arguments 
training_args = TrainingArguments(
    per_device_train_batch_size=1,  # Small batch size to fit in memory
    gradient_accumulation_steps=4,  # Accumulate gradients for effective batch size of 4
    num_train_epochs=5,  
    learning_rate=1e-4,  
    fp16=not torch.cuda.is_bf16_supported(),  # Use FP16 if BF16 not supported
    bf16=torch.cuda.is_bf16_supported(),  # Use BF16 if available (better precision)
    logging_steps=5,  # Log every 5 steps for frequent updates
    output_dir=output_dir,
    save_strategy="steps",  # Save every few steps instead of epochs
    save_steps=50,  # Save every 50 steps
    report_to="none",  # No external logging
    save_total_limit=2,  # Keep only last 2 checkpoints
    max_steps=500,  # Limit steps to prevent long runs; adjust based on dataset
    max_grad_norm=1.0,  # Gradient clipping
    gradient_checkpointing=True,  # Saves memory by recomputing activations
    optim="paged_adamw_8bit",  # Memory-efficient optimizer
    dataloader_num_workers=0,  # Disable multiprocessing to avoid issues
    dataloader_pin_memory=False,  # Disable pinning for stability
)

# SFTTrainer for instruction tuning
# Use the default columns: 'prompt' and 'completion' (created in preprocessing)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
)

In [ ]:
# 6. Training Loop

# Clear CUDA cache before training to avoid OOM
import torch
import gc

# Function to log GPU memory usage
def log_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

# Clear cache and log initial memory
torch.cuda.empty_cache()
gc.collect()
log_memory()

# Train the model
print("Starting training...")
trainer.train()

# Log memory after training
log_memory()

# Save LoRA adapter
trainer.model.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

In [ ]:
# 7. Inference Test: Generate Manim Code

def generate_manim_code(prompt, model, tokenizer, max_new_tokens=512):
    model.eval()
    input_text = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response (after [/INST])
    if '[/INST]' in decoded:
        decoded = decoded.split('[/INST]', 1)[-1].strip()
    return decoded

# Example prompt
test_prompt = "Create a Manim animation explaining Pythagoras theorem"
response = generate_manim_code(test_prompt, trainer.model, tokenizer)
print("Prompt:\n", test_prompt)
print("\nGenerated Manim code:\n", response)

# Save output for inspection
os.makedirs("/kaggle/working/outputs/mistral", exist_ok=True)
with open("/kaggle/working/outputs/mistral/test_output.py", "w", encoding="utf-8") as f:
    f.write(response)
print("\nSample output saved to /kaggle/working/outputs/mistral/test_output.py")

In [ ]:
# 8. Optional: Early Stopping and Basic Metrics
# This cell adds validation and early stopping to prevent overfitting.
# Note: Metrics calculation can be memory-intensive; skip if training freezes.

from transformers import EarlyStoppingCallback, TrainerCallback
from evaluate import load                                        
import matplotlib.pyplot as plt
import os

# Add validation split (10% of data for evaluation)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
val_dataset = dataset_split['test']

# Simple compute_metrics function (ROUGE only for simplicity)
rouge = load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {'rougeL': rouge_result['rougeL'].mid.fmeasure}

# Early stopping to stop if no improvement
early_stopping = EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)

# Custom callback to log losses for plotting
class MetricsLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_loss = []
        self.eval_loss = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_loss.append(logs['loss'])
            if 'eval_loss' in logs:
                self.eval_loss.append(logs['eval_loss'])

metrics_logger = MetricsLoggerCallback()

# Re-create trainer with validation and callbacks
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    args=training_args,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping, metrics_logger],
)

# Train with validation
trainer.train()

# Save LoRA adapter
trainer.model.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

# Plot basic loss curves (if training completed)
if metrics_logger.train_loss and metrics_logger.eval_loss:
    os.makedirs('/kaggle/working/outputs/mistral/metrics', exist_ok=True)
    plt.figure(figsize=(8,5))
    plt.plot(metrics_logger.train_loss, label='Train Loss')
    plt.plot(metrics_logger.eval_loss, label='Validation Loss')
    plt.xlabel('Logging Step')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.savefig('/kaggle/working/outputs/mistral/metrics/loss_curve.png')
    plt.close()
    print('Loss curve saved to /kaggle/working/outputs/mistral/metrics/loss_curve.png')
else:
    print("No metrics logged; training may have stopped early.")

In [ ]:
import os
import zipfile

# Directory where Kaggle saves notebook outputs
output_dir = "/kaggle/working"

# Name of zip file
zip_file = "kaggle_output.zip"

# Create zip
with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            z.write(file_path, arcname=os.path.relpath(file_path, output_dir))

print("ZIP created successfully:", zip_file)